# 02 · Similarity: how each dimension is scored

**Purpose.** Decide how to score a marketed listing against the portfolio on each of the three dimensions the brief
names: **age** (section 1), **location** (section 2) and **physical** characteristics (section 3). Each section
looks at the data, compares ways of scoring it, and ends with its decisions for `src/indurent_matching/similarity.py`.

Each dimension gets two kinds of score: **pairwise similarity** (how similar the listing is to one portfolio unit, for
prompt 1) and **homogeneity** (how typical the listing is of the portfolio, for prompt 2).


## 1. Age (build year)

**Purpose.** Decide how to score the **age** dimension: how similar a marketed listing's build year is to the
portfolio's, as one input to the brief's prompt 1 (the most similar properties) and for prompt 2 (a homogeneity score
for each dimension, age among them).

The data is the cleaned output of `indurent_matching.data` (see `01_eda.ipynb` for the cleaning decisions).
The section ends with decisions to carry into `src/indurent_matching/similarity.py`.

### 1.1 The question

**Age similarity** (prompt 1) asks: *how close is each portfolio unit's build year to the listing's?* It gives one
score per unit, which goes into the ranking of the most similar properties.

**Age homogeneity** (prompt 2) asks: *is this listing's build year typical of the portfolio?* A listing built in a year
when Indurent owns plenty of stock is homogeneous with it on age; one built in a year it hardly owns is not.

We start with build year because it's the most straightforward: one feature, recorded for almost every unit.
The method below is chosen for the shape of the build-year data, not assumed in advance.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

from indurent_matching.data import load_marketed, load_portfolio

# Store each chart's data in the notebook itself; JupyterLab draws it with the Plotly extension, and the HTML
# copy is a fallback for other viewers.
pio.renderers.default = "plotly_mimetype+notebook_connected"

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
portfolio = load_portfolio(ROOT / "data" / "Current Portfolio.xlsx")
marketed = load_marketed(ROOT / "data" / "Marketed Warehouses.xlsx")

# Portfolio build years, leaving out any flagged doubtful (none are at present).
years = portfolio.loc[~portfolio["build_year_doubtful"], "build_year"].dropna()
print(f"Portfolio: {portfolio['build_year'].notna().sum():,} of {len(portfolio):,} units have a build year; "
      f"leaving out {portfolio['build_year_doubtful'].sum()} flagged doubtful gives {len(years):,}")
marketed[["industrial_estate_name", "unit_name", "build_year"]]

Portfolio: 1,234 of 1,250 units have a build year; leaving out 0 flagged doubtful gives 1,234


,industrial_estate_name,unit_name,build_year
listing_id,,,
M1,Cherry Lane,Unit 10,1977.0
M2,Tech Hub,Unit 8,NaN
M3,Spitfire Park,Unit 42,1976.0
M4,Stable Lane,Unit 14,2000.0
M5,Chancery Depot,Unit 2,1966.0


* **No build years are flagged doubtful.** Granite Mews 59's (1939-1953) were suspected at first (EDA 2.9), but
  a 1940s industrial building is plausible, and the EPC A ratings that seemed to contradict them can't be relied
  on. Only its eaves are flagged. Any build year flagged in future would be left out here.
* Four of the five listings have a build year. **M2 (Tech Hub) has none**, so its age score will be "not
  available", not a low score.

### 1.2 Pairwise similarity (prompt 1)

**`age_similarity(listing, portfolio)`** scores every portfolio unit by the gap between its build year and the
listing's, using **`halving(gap, half_life)`**: 1 for the same year, halving with every `AGE_HALF_LIFE_YEARS` (10)
years of gap. The running example is M3 (Spitfire Park, built 1976). First, what `halving` gives for a few gaps:


In [2]:
from indurent_matching.similarity import age_similarity, halving

m3 = marketed.loc["M3"]
gaps = pd.Series([0, 5, 10, 20, 40, -10])
pd.DataFrame({"gap": gaps, "halving(gap, 10)": halving(gaps, 10).round(3)})


,gap,"halving(gap, 10)"
0,0,1.000
1,5,0.707
2,10,0.500
3,20,0.250
4,40,0.062
5,-10,0.500


0.5 at one half-life, 0.25 at two, and the same for -10 as for +10: direction doesn't matter. Applied to the gap
between M3 (1976) and every portfolio unit:

In [3]:
m3_age = age_similarity(m3, portfolio)
print(f"{len(m3_age):,} scores, one per portfolio unit; {m3_age.isna().sum()} are NaN "
      "(units with no usable build year)")

fig = px.scatter(x=portfolio["build_year"], y=m3_age, height=360, opacity=0.4,
                 title="Age similarity of every portfolio unit to M3 (built 1976)",
                 labels={"x": "Portfolio unit build year", "y": "Age similarity to M3"})
fig.add_vline(x=1976, line_dash="dash", line_color="firebrick")
fig.show()

1,250 scores, one per portfolio unit; 16 are NaN (units with no usable build year)


In [4]:
sample = portfolio.loc[[*m3_age.dropna().sort_values().index[[0, 300, 700, 1000, -1]]],
                       ["industrial_estate_name", "unit_name", "build_year"]]
sample["gap (years)"] = (sample["build_year"] - 1976).abs()
sample["age_similarity to M3"] = m3_age[sample.index].round(2)
sample

,industrial_estate_name,unit_name,build_year,gap (years),age_similarity to M3
property_id,,,,,
895,Sable Crescent 13,Unit 2,2022.0,46.0,0.04
1210,Umber Mews 9,Unit 1,2009.0,33.0,0.10
552,Teal Mews 69,Unit 3,1993.0,17.0,0.31
682,Ruby Junction 67,Unit 16,1982.0,6.0,0.66
969,Topaz Park 9,Unit 3,1976.0,0.0,1.00


* Every unit sits on one curve: 1 for units built in 1976, falling either side, and 0.5 at 1966 and 1986.
* The 16 units without a build year are NaN. They're left as NaN here; when the
  dimensions are combined for the ranking, a unit's NaN counts as 0.
* For M2 every score would be NaN, because M2 has no build year.


### 1.3 Homogeneity (prompt 2): the distribution of build years

In [5]:
fig = px.histogram(years, x="build_year", nbins=int(years.max() - years.min() + 1),
                   title="Portfolio build years (dashed lines: marketed listings' build years)",
                   labels={"build_year": "Build year"}, height=420)
fig.update_layout(showlegend=False, yaxis_title="Units")
for year in marketed["build_year"].dropna():
    fig.add_vline(x=year, line_dash="dash", line_color="firebrick")
fig.show()

* From **1970 to 2021 the distribution is roughly flat**: no decade dominates, and no year stands out apart from
  ordinary year-to-year noise.
* Before 1970 there's a **thin tail** back to 1939.
* **2022 is a spike**, the tallest bar by far and the last year in the data (the 5-year bands below look at it).
* The listings: **M1 (1977), M3 (1976) and M4 (2000)** sit in the flat part, among plenty of portfolio stock.
  **M5 (1966)** is in the thin tail.

In [6]:
per_year = years.value_counts().sort_index()
periods = pd.cut(years, [1900, 1970, 1980, 1990, 2000, 2010, 2020, 2022, 2023], right=False,
                 labels=["before 1970", "1970s", "1980s", "1990s", "2000s", "2010s", "2020-21", "2022"])
summary = years.groupby(periods, observed=False).size().to_frame("units")
summary["years covered"] = [int(1970 - years.min()), 10, 10, 10, 10, 10, 2, 1]
summary["units per year"] = (summary["units"] / summary["years covered"]).round(1)
summary

,units,years covered,units per year
build_year,,,
before 1970,46,31,1.5
1970s,202,10,20.2
1980s,217,10,21.7
1990s,251,10,25.1
2000s,232,10,23.2
2010s,175,10,17.5
2020-21,31,2,15.5
2022,80,1,80.0


* **18-25 units a year** in every decade from the 1970s to the 2010s: close to uniform, as the histogram
  suggests. The 2010s and 2020-21 are slightly lower, which fits newer stock taking time to be acquired.
* **Before 1970: about 1.5 units a year** (46 units, 1939-1969), so a pre-1970 listing has few portfolio peers
  on age.
* **2022: 80 units**, about four times a normal year. Real recent stock, or a default year? The 5-year bands below come back to it.
* So "homogeneous on age" should mean roughly: *any year from 1970 to 2021 is typical, earlier years are
  increasingly unusual*. The scoring method has to respect that shape: a flat middle, a thin tail and a spike.

**Kernel density smoothing.** A kernel density estimate (KDE) places a small bell curve on every unit's build
year and adds them up, giving a smooth curve of how common each year is. The width of each bell (the
*bandwidth*) sets how much it smooths; `scipy` picks it from the data (Scott's rule).

In [7]:
import plotly.graph_objects as go
from scipy.stats import gaussian_kde

kde = gaussian_kde(years)
bandwidth = kde.factor * years.std()
grid = np.linspace(years.min() - 10, years.max() + 10, 400)

fig = go.Figure()
fig.add_histogram(x=years, xbins={"size": 1}, histnorm="probability density", name="build years",
                  marker_color="lightsteelblue")
fig.add_scatter(x=grid, y=kde(grid), mode="lines", name=f"KDE (bandwidth {bandwidth:.1f} years)",
                line={"color": "navy", "width": 2})
for year in marketed["build_year"].dropna():
    fig.add_vline(x=year, line_dash="dash", line_color="firebrick")
fig.update_layout(title="Portfolio build years with kernel density smoothing (dashed lines: listings)",
                  xaxis_title="Build year", yaxis_title="Share of units per year", height=420)
fig.show()
print(f"Bandwidth: {bandwidth:.1f} years")

Bandwidth: 4.0 years


* With a **bandwidth of about 4 years**, the curve follows the overall shape: the flat 1970-2021 middle, at
  roughly 1.5-2% of units per year, and the thin pre-1970 tail, falling to about 0.5% a year around 1966 (M5).
* It irons out year-to-year noise, which is what we want: a listing built in 1977 shouldn't score differently
  from one built in 1978 just because one year happened to have a few more units.
* Two problems at the **right-hand edge**:
  * the **2022 spike is smoothed away**: 6.5% of units are recorded as 2022, but the curve gives it about 1.3%,
    spreading the spike over the years either side, including years after 2022;
  * the curve **spills past 2022** into years with no buildings (still about 0.9% a year at 2024), and dips
    at 2022 itself, because there's no data beyond the edge to balance it.
* Both matter only for listings built near 2022 (none of ours), but a scoring method should handle the edge
  properly. The next two charts look at fixing it.

**Frequency per year, scaled so the maximum is 1.** A simpler alternative that has no edge problem: count the
portfolio units built in each year, and divide by the count in the most frequent year. A listing scores the value for
its build year: 1 means "built in the portfolio's most common year".

In [8]:
per_year = years.astype(int).value_counts().reindex(range(int(years.min()), int(years.max()) + 1), fill_value=0)
year_score = per_year / per_year.max()

fig = px.bar(x=year_score.index, y=year_score.values, height=380,
             title="Score per build year: units that year / units in the most frequent year (dashed lines: listings)",
             labels={"x": "Build year", "y": "Score"})
for year in marketed["build_year"].dropna():
    fig.add_vline(x=year, line_dash="dash", line_color="firebrick")
fig.show()

print(f"Most frequent year: {per_year.idxmax()} ({per_year.max()} units). "
      f"1970-2021 scores: median {year_score.loc[1970:2021].median():.2f}, "
      f"range {year_score.loc[1970:2021].min():.2f}-{year_score.loc[1970:2021].max():.2f}")
listed = marketed[["build_year"]].dropna().astype(int)
listed["units that year"] = listed["build_year"].map(per_year)
listed["score"] = listed["build_year"].map(year_score).round(2)
listed

Most frequent year: 2022 (80 units). 1970-2021 scores: median 0.28, range 0.12-0.41


,build_year,units that year,score
listing_id,,,
M1,1977,30,0.38
M3,1976,24,0.30
M4,2000,25,0.31
M5,1966,8,0.10


* **No edge problem**: nothing spills past 2022, and 2022 keeps its full count.
* But **2022 sets the scale for every other year.** With 80 units it scores 1, so the whole flat 1970-2021
  middle is squeezed into 0.12-0.41 (a typical year scores about 0.28). A 1990s listing would read as "not very
  homogeneous" although Indurent owns plenty of 1990s stock, and if 2022 turns out to be a default year, every
  score is distorted by it.
* **Year-to-year noise shows through**: 1986, 1987 and 1988 score 0.28, 0.41 and 0.28, so a 1987 listing would
  look 50% more homogeneous than a 1986 one by chance. The KDE smoothed that away.
* The listings: M5 (1966) 0.10, M3 (1976) 0.30, M1 (1977) 0.38, M4 (2000) 0.31. M1 beats M3 only because
  1977 happens to have six more units than 1976.

**Frequency per 5-year band, scaled so the maximum is 1.** The same idea with wider bins: counting over five
years averages out the noise, and a one-year spike lifts its own band instead of setting the scale for all of
them.

In [9]:
band_start = years.astype(int) // 5 * 5
per_band = band_start.value_counts().sort_index()
band_score = per_band / per_band.max()
band_label = {start: f"{start}-{str(min(start + 4, int(years.max())))[2:]}" for start in per_band.index}

fig = px.bar(x=[band_label[s] for s in per_band.index], y=band_score.values, height=380,
             title="Score per 5-year band: units in the band / units in the most frequent band",
             labels={"x": "Build year band", "y": "Score"})
fig.update_xaxes(type="category")  # otherwise labels like "1975-79" are read as dates
fig.show()

listed["band"] = (listed["build_year"] // 5 * 5).map(band_label)
listed["band score"] = (listed["build_year"] // 5 * 5).map(band_score).round(2)
listed[["build_year", "band", "band score"]]

,build_year,band,band score
listing_id,,,
M1,1977,1975-79,0.86
M3,1976,1975-79,0.86
M4,2000,2000-04,0.86
M5,1966,1965-69,0.20


In [10]:
# Is the jump between the 1965-69 and 1970-74 bands real? Look at the single years either side.
per_year.loc[1965:1974].to_frame("units").T

build_year,1965,1966,1967,1968,1969,1970,1971,1972,1973,1974
units,4,8,3,5,5,20,17,18,16,22


* **1970-2022 scores 0.6-1.0**; the most frequent band is 1995-99 (127 units). The 1960s score 0.05-0.20, and
  the bands before 1960 0.08 or less.
* **2022 no longer dominates**: it lifts the 2020-22 band to 0.87, among the most typical, but not above the
  rest. The last band covers only three years (there's nothing after 2022), so its count includes the spike.
* The listings: **M1 and M3 now score the same (0.86)**, as they should for buildings a year apart; **M4 also
  0.86**; **M5 0.20**. M2 has no build year, so no score.
* **The jump between 1965-69 (0.20) and 1970-74 (0.73) is in the data, not made by the bands.** The single-year
  counts jump at the same place: 1965-1969 each have a handful of units, 1970 onwards around 20 a year. So
  Indurent's stock really does start in earnest in 1970, and a 1969 building really is rarer in the portfolio
  than a 1970 one.

### 1.4 Decisions for `similarity.py`

We are going to use 5 year bands, rescaled so that the max of the bands has a score of 1

| Topic | Decision | Why |
|---|---|---|
| Age score | Units in the listing's 5-year build-year band, divided by units in the most frequent band (0-1) | Frequency has no edge problem at 2022; 5-year bands average out year-to-year noise and stop the 2022 spike setting the scale |
| Bands | Fixed 5-year bands (1935-39, ..., 2020-22) | The jump at 1970 is in the data, not made by the bands |
| Doubtful build years | Left out of the counts (none are flagged at present) | Doubtful values are never scored. Granite Mews 59's 1939-1953 build years were suspected but are plausible, so they count |
| No build year | Age score "not available" (M2), never 0 | A missing value says nothing about age |
| Not used | Kernel density smoothing; single-year frequency | KDE spills past 2022 and smooths away the spike; single years are noisy and let 2022 set the scale |
| Not used | A moving 5-year window centred on the listing's year (the method used for size, section 3) | The fullest window would be 2018-22 (145 units) because it captures the whole 2022 spike, so the spike would set the scale and typical 1970s-2000s listings would drop from 0.86 to about 0.75. Fixed bands keep the spike in a 3-year band (2020-22, 111 units), below the most frequent band (1995-99, 127). Size has no spike and slopes smoothly, so there the moving window is better. If the 2022 spike turned out to be a default year (and was flagged), the moving window would suit age too |


## 2. Location

**Purpose.** Decide how to score the **location** dimension: how well a marketed listing's location fits the
portfolio, as one input to the brief's prompt 1 (the most similar properties) and for prompt 2 (a homogeneity score
for each dimension, location among them).

The data is the cleaned output of `indurent_matching.data` (see `01_eda.ipynb` for the cleaning decisions,
EDA section 2.2 for the location checks). The section ends with decisions to carry into
`src/indurent_matching/similarity.py`.

### 2.1 The question and the usable data

**Location similarity** (prompt 1) asks: *how close is each portfolio unit to the listing on the map?* It gives one
score per unit, which goes into the ranking of the most similar properties.

**Location homogeneity** (prompt 2) asks: *does Indurent own stock near this listing?* A listing in an area where the
portfolio is dense is homogeneous with it on location; one far from any portfolio unit is not.

Only locations we trust are used: **high** confidence (inside the labelled region) and **medium** (on land, but
outside the labelled region). **Unknown** locations (no coordinates, or 1 km+ offshore) have no location score.

In [11]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

from indurent_matching.data import load_marketed, load_portfolio

# Store each chart's data in the notebook itself; JupyterLab draws it with the Plotly extension, and the HTML
# copy is a fallback for other viewers.
pio.renderers.default = "plotly_mimetype+notebook_connected"

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
portfolio = load_portfolio(ROOT / "data" / "Current Portfolio.xlsx")
marketed = load_marketed(ROOT / "data" / "Marketed Warehouses.xlsx")

located = portfolio[portfolio["location_confidence"] != "unknown"]
print(f"Portfolio: {len(located):,} of {len(portfolio):,} units have a usable location, "
      f"in {located['industrial_estate_name'].nunique()} of {portfolio['industrial_estate_name'].nunique()} estates")
portfolio["location_confidence"].value_counts().to_frame("units")

Portfolio: 892 of 1,250 units have a usable location, in 151 of 263 estates


,units
location_confidence,
medium,652
unknown,358
high,240


In [12]:
marketed[["industrial_estate_name", "region", "location_confidence", "derived_region", "nearest_city",
          "miles_to_city"]].round(1)

,industrial_estate_name,region,location_confidence,derived_region,nearest_city,miles_to_city
listing_id,,,,,,
M1,Cherry Lane,North West,medium,Wales,Cardiff,72.8
M2,Tech Hub,South East,unknown,None,NaN,NaN
M3,Spitfire Park,Midlands,high,Midlands,Manchester,14.0
M4,Stable Lane,South East,medium,East of England,London,40.6
M5,Chancery Depot,NaN,unknown,None,NaN,NaN


* **892 of 1,250 portfolio units** (151 of 263 estates) have a usable location: 240 high confidence, 652 medium.
  The other 358 can't take part in location comparisons, either as matches or in the counts a listing is
  scored against.
* Three listings have a usable location:
  * **M1 (Cherry Lane)**: medium; labelled North West but its coordinates fall in **Wales**.
  * **M3 (Spitfire Park)**: high; Midlands, 14 miles from Manchester.
  * **M4 (Stable Lane)**: medium; labelled South East but its coordinates fall in the **East of England**.
* **M2 (Tech Hub) has no coordinates and M5 (Chancery Depot) is 27 km offshore**, so their location score is
  "not available", not a low score.

### 2.2 Where the portfolio is

In [13]:
fig = px.scatter_geo(located, lat="latitude", lon="longitude", color="location_confidence",
                     hover_name="industrial_estate_name", hover_data=["unit_name", "derived_region"],
                     height=750, title="Portfolio units with a usable location, and the listings (stars)")
fig.update_traces(marker={"size": 5, "opacity": 0.6})
listings = marketed[marketed["location_confidence"] != "unknown"]
fig.add_scattergeo(lat=listings["latitude"], lon=listings["longitude"], name="marketed listing",
                   text=listings["listing_id"] + " " + listings["industrial_estate_name"], hoverinfo="text",
                   marker={"symbol": "star", "size": 16, "color": "firebrick", "line": {"width": 1, "color": "white"}})
fig.update_geos(resolution=50, showland=True, landcolor="#f2f2f2", showcountries=True,
                lataxis_range=[49.8, 59], lonaxis_range=[-7, 2.2])
fig.show()

In [14]:
by_region = located.groupby("derived_region").agg(units=("property_id", "size"),
                                                   estates=("industrial_estate_name", "nunique"))
by_region["% of units"] = (by_region["units"] / by_region["units"].sum() * 100).round(0)
by_region.sort_values("units", ascending=False)

,units,estates,% of units
derived_region,,,
Midlands,279,41,31.0
East of England,175,28,20.0
South West,107,20,12.0
South East,96,18,11.0
North West,84,11,9.0
Wales,82,7,9.0
North East,26,8,3.0
Yorks. & Humber,24,11,3.0
London,11,3,1.0


* The portfolio is **concentrated in the Midlands (31% of located units) and the East of England (20%)**, with
  the South West, South East, North West and Wales at 9-12% each. The North East, Yorkshire, London and
  Scotland together hold under 10%.
* By region, the listings sit in quite different places:
  * **M3 (Midlands)** is in the portfolio's heartland;
  * **M4 (East of England)** is in its second-biggest region;
  * **M1 (Wales)** is in a region with 82 units but only 7 estates.
* Regions are coarse, though: a listing can be in a busy region but far from any unit in it. The score should
  use actual distances (section 2.3), not region membership.

In [15]:
estate_sizes = located["industrial_estate_name"].value_counts()
print(f"Largest estates: {', '.join(f'{name} ({n})' for name, n in estate_sizes.head(5).items())}")
print(f"The 5 largest estates hold {estate_sizes.head(5).sum() / len(located):.0%} of located units; "
      f"{(estate_sizes == 1).sum()} estates are a single unit")

Largest estates: Quartz Wharf 16 (64), Ruby Arc 96 (48), Velvet Park 60 (41), Granite Pointe 39 (37), Crimson Forge 9 (37)
The 5 largest estates hold 25% of located units; 57 estates are a single unit


* **A few large estates hold a big share of the units**: the 5 largest hold a quarter of all located units
  (Quartz Wharf 16 alone has 64), while 57 estates are a single unit.
* That matters for any score built by counting nearby units: a listing next to Quartz Wharf 16 would look very
  homogeneous on location because of one estate, much as the 2022 spike inflated one build year. Counting
  **estates** instead of units, or both, is worth comparing when we build the score (section 2.5).

### 2.3 Each listing sits on an existing estate

Before scoring, a check: how close is each listing to the nearest portfolio unit, and which estate is it in?
This uses every listing with coordinates, including M5 (offshore), since the question is only which portfolio
units share its coordinates.

In [16]:
from indurent_matching.geo import haversine_miles

with_coords = portfolio.dropna(subset=["latitude", "longitude"])
rows = []
for listing_id, listing in marketed.dropna(subset=["latitude", "longitude"]).iterrows():
    miles = haversine_miles(listing["latitude"], listing["longitude"], with_coords["latitude"], with_coords["longitude"])
    host = with_coords.loc[miles.idxmin(), "industrial_estate_name"]
    host_units = portfolio[portfolio["industrial_estate_name"] == host]
    rows.append({
        "listing": f"{listing_id} {listing['industrial_estate_name']}",
        "listing unit": listing["unit_name"],
        "metres to nearest portfolio unit": round(miles.min() * 1609),
        "sits on estate": host,
        "units in estate": len(host_units),
        "listing region label": listing["region"],
        "estate region label": host_units["region"].iloc[0],
        "location confidence": listing["location_confidence"],
    })
pd.DataFrame(rows).set_index("listing")

,listing unit,metres to nearest portfolio unit,sits on estate,units in estate,listing region label,estate region label,location confidence
listing,,,,,,,
M1 Cherry Lane,Unit 10,40,Scarlet Pavilion 78,9,North West,North West,medium
M3 Spitfire Park,Unit 42,9,Velvet Park 60,41,Midlands,Midlands,high
M4 Stable Lane,Unit 14,21,Sable Mill 91,10,South East,South East,medium
M5 Chancery Depot,Unit 2,71,Granite Depot 69,1,NaN,Wales,unknown


* **Every listing with coordinates sits on an existing Indurent estate**: its nearest portfolio unit is within
  about 50 m, the same spread as units within one estate.
* The **unit numbers follow on from the estate's own**: Velvet Park 60 has Units 1-41 and M3 is Unit 42;
  Scarlet Pavilion 78 has 1-9 and M1 is Unit 10; Granite Depot 69 has Unit 1 and M5 is Unit 2. (M4 is Unit 14
  on an estate of 10.)
* The **region labels match the estate's too**, including the wrong ones: M1 and Scarlet Pavilion 78 are both
  labelled North West but sit in Wales.
* **M5's estate, Granite Depot 69, has the same offshore coordinates**, so M5 stays "not available" for location.
* So these listings look like **further units on estates Indurent already owns** (infill acquisitions). That's
  the single most useful location fact for an answer: *"M3 sits on Velvet Park 60, an estate you already own."*
* It also means **nearest portfolio units are the wrong neighbours**: they would all be from the listing's own
  estate, at about 0 miles, so every listing would get the same perfect score. Section 2.5 uses **estates** as the
  neighbours instead, and leaves out the estate the listing sits on.

### 2.4 Pairwise similarity (prompt 1)

**`location_similarity(listing, portfolio)`** scores every portfolio unit by its straight-line miles from the
listing, halving with every `LOCATION_HALF_LIFE_MILES` (10) miles. For M3, on the map:


In [17]:
from indurent_matching.similarity import location_similarity

m3 = marketed.loc["M3"]
m3_location = location_similarity(m3, portfolio)
print(f"{m3_location.notna().sum()} units scored; {m3_location.isna().sum()} NaN (no usable location). "
      f"Units scoring 0.5 or more (within 10 miles): {(m3_location >= 0.5).sum()}")

scored = portfolio.copy()
scored["similarity"] = m3_location
scored = scored.dropna(subset=["similarity"])
fig = px.scatter_geo(scored, lat="latitude", lon="longitude", color="similarity", color_continuous_scale="Viridis",
                     hover_name="industrial_estate_name", height=650,
                     title="Location similarity of every portfolio unit to M3 (star)")
fig.update_traces(marker={"size": 6})
fig.add_scattergeo(lat=[m3["latitude"]], lon=[m3["longitude"]], name="M3", showlegend=False,
                   marker={"symbol": "star", "size": 16, "color": "firebrick"})
fig.update_geos(resolution=50, showland=True, landcolor="#f2f2f2", showcountries=True,
                lataxis_range=[49.8, 59], lonaxis_range=[-7, 2.2])
fig.show()

892 units scored; 358 NaN (no usable location). Units scoring 0.5 or more (within 10 miles): 46


In [18]:
nearest = scored.sort_values("similarity", ascending=False)
by_estate = nearest.groupby("industrial_estate_name", sort=False).agg(
    units=("unit_name", "size"), similarity=("similarity", "max")).head(6)
by_estate["miles"] = (-10 * np.log2(by_estate["similarity"])).round(1)  # back from similarity to miles
by_estate.round(2)

,units,similarity,miles
industrial_estate_name,,,
Velvet Park 60,41,1.00,0.0
Jade Bridge 6,4,0.53,9.1
Indigo Wharf 90,1,0.51,9.6
Azure Summit 81,1,0.34,15.4
Scarlet Arc 73,1,0.33,16.2
Cedar Pavilion 55,8,0.29,17.7


* Units near M3 score close to 1 and the score fades with distance; most of the portfolio is far enough away to
  score near 0.
* The **41 units of Velvet Park 60**, the estate M3 sits on, all score about 1. That's why, without a cap, M3's
  top 10 for prompt 1 can be all Velvet Park 60: on location alone they're unbeatable.
* The next estates (Jade Bridge 6 at about 9 miles, and so on) score around 0.5.
* Units with no usable location are NaN here, and count as 0 when the dimensions are combined.

### 2.5 Distance to the nearest estates

Each estate becomes one point (the average of its units' coordinates, usable locations only). For a listing,
we measure the distance to its **k nearest estates, leaving out the one it sits on**; for comparison, each
portfolio estate is measured against its k nearest *other* estates.

In [19]:
estates = located.groupby("industrial_estate_name")[["latitude", "longitude"]].mean()


def nearest_estates(lat, lon, exclude, k):
    """Distances in miles to the k nearest estates, leaving out ``exclude``."""
    others = estates.drop(exclude)
    return np.sort(haversine_miles(lat, lon, others["latitude"], others["longitude"]))[:k]


hosts = pd.DataFrame(rows).assign(listing_id=lambda t: t["listing"].str.split().str[0]).set_index("listing_id")
usable = marketed[marketed["location_confidence"] != "unknown"]
table = pd.DataFrame({
    f"mean miles, {k} nearest": [
        nearest_estates(r["latitude"], r["longitude"], hosts.loc[i, "sits on estate"], k).mean()
        for i, r in usable.iterrows()
    ]
    for k in [1, 5, 10]
}, index=usable.index).round(1)
table.insert(0, "sits on estate", hosts.loc[usable.index, "sits on estate"])
table

,sits on estate,"mean miles, 1 nearest","mean miles, 5 nearest","mean miles, 10 nearest"
listing_id,,,,
M1,Scarlet Pavilion 78,4.4,36.2,48.2
M3,Velvet Park 60,9.1,13.6,20.3
M4,Sable Mill 91,6.3,11.5,17.3


* **k matters, mostly for M1.** Its single nearest other estate is only 4.4 miles away (the closest of the
  three), but its 5 nearest average 36 miles: one neighbour, then a big gap. With k = 1, M1 would look like
  the best-placed listing; with k = 5 or 10, the most isolated.
* **k = 1 is fragile**: one estate decides the score. **k = 5 or 10** describe the surrounding area, which is
  what "homogeneous on location" should mean. M3 and M4 rank the same either way.

In [20]:
K = 5  # decided: mean of the 5 nearest other estates
portfolio_mean = pd.Series(
    [nearest_estates(r["latitude"], r["longitude"], name, K).mean() for name, r in estates.iterrows()],
    index=estates.index,
)
fig = px.histogram(portfolio_mean, nbins=40, height=380,
                   title=f"Portfolio estates: mean miles to their {K} nearest other estates (dashed lines: listings)",
                   labels={"value": "Mean miles"})
fig.update_layout(showlegend=False, yaxis_title="Estates")
for miles in table[f"mean miles, {K} nearest"]:
    fig.add_vline(x=miles, line_dash="dash", line_color="firebrick")
fig.show()
print(f"Portfolio estates: median {portfolio_mean.median():.1f} miles, "
      f"10th-90th percentile {portfolio_mean.quantile(0.1):.1f}-{portfolio_mean.quantile(0.9):.1f}, "
      f"max {portfolio_mean.max():.1f} ({portfolio_mean.idxmax()})")

Portfolio estates: median 16.1 miles, 10th-90th percentile 11.1-31.9, max 126.8 (Sable Forge 35)


* A typical portfolio estate has its 5 nearest other estates **about 16 miles away on average**; 80% fall
  between 11 and 32 miles. A long tail of isolated estates runs out to 127 miles (Sable Forge 35).
* **M4 (11.5 miles) and M3 (13.6 miles)** are in typical-to-dense parts of the portfolio, besides sitting on an
  estate Indurent owns.
* **M1 (36.2 miles)** is beyond 90% of portfolio estates: apart from Scarlet Pavilion 78, the estate it sits on,
  and one other estate 4 miles away, Indurent owns little nearby.

### 2.6 Homogeneity (prompt 2): the location score

**Decision:** the score uses the **mean distance to the 5 nearest other estates**, rescaled so that the
best-placed estate in the portfolio scores 1:

score = (best portfolio estate's mean distance) / (listing's mean distance)

The score halves each time the distance doubles: 0.5 means the listing's nearest estates are on average twice as
far away as for the best-placed estate. This matches the age score, where the most frequent band scores 1. The nearest
other estate is reported alongside as context, not scored.

In [21]:
best = portfolio_mean.min()
portfolio_score = best / portfolio_mean
print(f"Best-placed estate: {portfolio_mean.idxmin()} ({best:.1f} miles to its 5 nearest on average). "
      f"Portfolio scores: median {portfolio_score.median():.2f}, "
      f"10th-90th percentile {portfolio_score.quantile(0.1):.2f}-{portfolio_score.quantile(0.9):.2f}")

scores = table[["sits on estate", "mean miles, 1 nearest", f"mean miles, {K} nearest"]].rename(
    columns={"mean miles, 1 nearest": "nearest other estate (miles)"})
scores["location score"] = (best / scores[f"mean miles, {K} nearest"]).round(2)
# Context, not scored: the share of portfolio estates the listing is better placed than.
scores["better placed than (% of estates)"] = [
    round((portfolio_score < s).mean() * 100) for s in best / scores[f"mean miles, {K} nearest"]
]
scores

Best-placed estate: Slate Hub 99 (8.5 miles to its 5 nearest on average). Portfolio scores: median 0.52, 10th-90th percentile 0.27-0.76


,sits on estate,nearest other estate (miles),"mean miles, 5 nearest",location score,better placed than (% of estates)
listing_id,,,,,
M1,Scarlet Pavilion 78,4.4,36.2,0.23,7
M3,Velvet Park 60,9.1,13.6,0.62,63
M4,Sable Mill 91,6.3,11.5,0.73,85


In [22]:
fig = px.histogram(portfolio_score, nbins=30, height=380,
                   title="Location score of portfolio estates (dashed lines: listings)",
                   labels={"value": "Location score"})
fig.update_layout(showlegend=False, yaxis_title="Estates")
for score in scores["location score"]:
    fig.add_vline(x=score, line_dash="dash", line_color="firebrick")
fig.show()

* The best-placed estate, **Slate Hub 99**, has its 5 nearest estates 8.5 miles away on average and scores 1.
  A typical portfolio estate scores **about 0.5**; 80% score between 0.27 and 0.76.
* The listings: **M4 0.73** and **M3 0.62**, both better placed than a typical portfolio estate; **M1 0.23**,
  in the bottom 10%.
* The **percentile** (share of portfolio estates the listing is better placed than) is context for an answer,
  not the score: it stretches small differences (M3 and M4 are 2 miles apart but would score 63% vs 85%).
* In an answer, the score comes with its context, e.g. *"M1 sits on Scarlet Pavilion 78, which you own. The
  next nearest estate is 4.4 miles away, but beyond that Indurent owns little nearby: location score 0.23,
  better placed than only 7% of Indurent's estates."*
* **M2 and M5: not available** (no coordinates; offshore).

### 2.7 Decisions for `similarity.py`

| Topic | Decision | Why |
|---|---|---|
| Usable locations | High and medium confidence only (892 units, 151 estates) | Unknown locations (no coordinates, or 1 km+ offshore) can't be placed (EDA 2.2) |
| Neighbours | Estates, not units; each estate is the average of its units' coordinates | Every listing sits on an existing estate, so the nearest units are all from that estate at about 0 miles, and every listing would score the same |
| The estate a listing sits on | Reported as a fact ("sits on Velvet Park 60, which you own"), left out of the score | The most useful location fact for an answer; leaving it out makes listings comparable with portfolio estates, which are measured against other estates |
| Distance | Mean miles to the 5 nearest other estates | Describes the surrounding area; a single nearest estate can mislead (M1: 4.4 miles to one, then 30+ miles to the rest) |
| Score | Best portfolio estate's mean distance / listing's mean distance (best = 1) | Same idea as the age score; halves each time the distance doubles; not set by the extreme isolated estates |
| Context, not scored | The nearest other estate (miles); the percentile (better placed than X% of portfolio estates) | Readable facts for answers; the percentile isn't the score because it stretches small differences |
| No usable location | Location score "not available" (M2, M5), never 0 | A missing location says nothing about where the property is |

## 3. Physical characteristics

**Purpose.** Decide how to score the **physical** dimension: how well a marketed listing's building fits the
portfolio, as one input to the brief's prompt 1 (the most similar properties) and for prompt 2 (a homogeneity score
for each dimension, physical among them).

The data is the cleaned output of `indurent_matching.data` (see `01_eda.ipynb` for the cleaning decisions).
The section ends with decisions to carry into `src/indurent_matching/similarity.py`.

**The marketed listings as supplied.** The raw file, before any cleaning, to show which fields there are to
score on. The physical fields are `Size sqm` (which holds square feet, EDA 2.1), `Min. Eaves m`, `Max. Eaves m`
and `Doors #`.

In [23]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
pd.read_excel(ROOT / "data" / "Marketed Warehouses.xlsx", dtype=object)

,Property Id,Industrial Estate Name,Unit Name,Region,Latitude,Longitude,Car Parking Spaces #,Size sqm,Build Year,Yard Depth m,Min. Eaves m,Max. Eaves m,Doors #,EPC Rating
0,NaN,Cherry Lane,Unit 10,North West,52.467186,-3.784003,30,16860.130016,1977,NaN,NaN,NaN,NaN,NaN
1,NaN,Tech Hub,Unit 8,South East,NaN,NaN,23,905.523948,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Spitfire Park,Unit 42,Midlands,53.505831,-1.905054,49,2294.49811,1976,NaN,NaN,NaN,NaN,B
3,NaN,Stable Lane,Unit 14,South East,52.078733,-0.349448,48,617.297521,2000,NaN,7,7,16,C
4,NaN,Chancery Depot,Unit 2,NaN,52.485703,-4.478974,22,16613.786473,1966,NaN,5.21,5.46,3,A


* **Size is the only physical field every listing has.**
* **Eaves (min and max) and doors are blank for M1, M2 and M3**; only M4 (Stable Lane) and M5 (Chancery Depot)
  have them, and M4's 16 doors on 617 sq ft were flagged as implausible in cleaning.
* **Yard depth is blank for all five**, which is one reason it isn't used for similarity (EDA 2.7).
* So whatever the portfolio holds, a physical comparison for three of the five listings can only rest on size.

### 3.1 The question and the usable data

**Physical similarity** (prompt 1) asks: *how alike are each portfolio unit's building and the listing's?* It
gives one score per unit, which goes into the ranking of the most similar properties.

**Physical homogeneity** (prompt 2) asks: *is this listing the kind of building Indurent owns?* The physical features are
size, eaves height (min and max) and doors. A value is usable only if it's recorded and not flagged doubtful;
doors also have to be the unit's own count, not an estate total copied onto every unit (EDA 2.6).

In [24]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

from indurent_matching.data import load_marketed, load_portfolio

# Store each chart's data in the notebook itself; JupyterLab draws it with the Plotly extension, and the HTML
# copy is a fallback for other viewers.
pio.renderers.default = "plotly_mimetype+notebook_connected"

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
portfolio = load_portfolio(ROOT / "data" / "Current Portfolio.xlsx")
marketed = load_marketed(ROOT / "data" / "Marketed Warehouses.xlsx")


def usable_physical(df):
    """The physical features with doubtful values, and door counts that aren't the unit's own, set to missing."""
    return pd.DataFrame({
        "size_sqft": df["size_sqft"].where(~df["size_sqft_doubtful"]),
        "min_eaves_m": df["min_eaves_m"].where(~df["min_eaves_m_doubtful"]),
        "max_eaves_m": df["max_eaves_m"].where(~df["max_eaves_m_doubtful"]),
        "doors": df["doors"].where(df["doors_describe_unit"] & ~df["doors_doubtful"]),
    })


physical = usable_physical(portfolio)
listing_physical = usable_physical(marketed)
pd.DataFrame({
    "portfolio units": physical.notna().sum(),
    "% of portfolio": (physical.notna().mean() * 100).round(0),
    "listings": listing_physical.notna().sum(),
})

,portfolio units,% of portfolio,listings
size_sqft,1240,99.0,5
min_eaves_m,409,33.0,2
max_eaves_m,270,22.0,2
doors,39,3.0,1


In [25]:
has = pd.DataFrame({
    "size": physical["size_sqft"].notna(),
    "eaves": physical[["min_eaves_m", "max_eaves_m"]].notna().any(axis=1),
    "doors": physical["doors"].notna(),
})
has.value_counts().rename("units").to_frame()

units
size  eaves doors       
True  False False    822
      True  False    379
            True      39
False False False      9
      True  False      1

In [26]:
listing_physical.round(1)

,size_sqft,min_eaves_m,max_eaves_m,doors
listing_id,,,,
M1,16860.1,NaN,NaN,NaN
M2,905.5,NaN,NaN,NaN
M3,2294.5,NaN,NaN,NaN
M4,617.3,7.0,7.0,NaN
M5,16613.8,5.2,5.5,3.0


* **Size is usable for almost every unit (1,240 of 1,250)**; the 10 without it are flagged under 500 sq ft.
* **Eaves for a third** (409 min, 270 max); **doors for only 39 units (3%)**, once estate totals and doubtful
  counts are left out.
* Most units can only be compared on size: **822 have size alone**, 379 size and eaves, 39 all three.
* The listings are the same: **M1, M2 and M3 have size only**; M4 has size and eaves (7 m; its 16 doors are
  flagged doubtful); M5 has size, eaves (5.2-5.5 m) and 3 doors.
* So **the physical dimension is mostly about size**, with eaves as a possible second feature. Doors are too sparse
  to matter: a listing's doors could be compared with only 39 portfolio units.

**What goes into the physical score**

| Feature | Decision | Why |
|---|---|---|
| Size | Scored, on a log scale | Usable for every listing and 99% of the portfolio |
| Eaves height | Context only, not scored (decided in section 3.5): the average of min and max, or whichever one is recorded | Clear height is what occupiers care about beyond floor area (how high they can rack), but only two listings have it. Using either side gives 419 units (260 both, 149 min only, 10 max only); where both exist they're close (median gap 1.1 m) |
| Doors | Context only, not scored | Only 39 portfolio units have a usable count of their own, and only M5 among the listings; counts rise with size (rank correlation 0.86), so they'd mostly repeat it |
| Volume (size x eaves) | Context only, not scored | 99% correlated with size (84% of its variation is size), needs eaves (419 units, two listings), and treats a big low shed and a small tall one as the same. Size and eaves scored separately capture both parts of it |

### 3.2 Size

In [27]:
size = physical["size_sqft"].dropna()
ticks = [500, 1_000, 2_000, 5_000, 10_000, 20_000, 50_000, 100_000, 200_000, 500_000, 1_000_000]

fig = px.histogram(x=np.log10(size), nbins=50, height=400,
                   title="Portfolio unit sizes, log scale (dashed lines: listings)",
                   labels={"x": "Size (sq ft, log scale)"})
fig.update_xaxes(tickvals=np.log10(ticks), ticktext=[f"{t:,}" for t in ticks])
fig.update_layout(yaxis_title="Units")
for sqft in listing_physical["size_sqft"].dropna():
    fig.add_vline(x=np.log10(sqft), line_dash="dash", line_color="firebrick")
fig.show()

In [28]:
print(f"Portfolio size: median {size.median():,.0f} sq ft; 10th-90th percentile "
      f"{size.quantile(0.1):,.0f}-{size.quantile(0.9):,.0f}; range {size.min():,.0f}-{size.max():,.0f}")
listing_size = listing_physical[["size_sqft"]].round(0)
listing_size["% of portfolio units smaller"] = [round((size < s).mean() * 100) for s in listing_size["size_sqft"]]
listing_size

Portfolio size: median 4,753 sq ft; 10th-90th percentile 1,235-44,393; range 505-1,195,451


,size_sqft,% of portfolio units smaller
listing_id,,
M1,16860.0,79
M2,906.0,5
M3,2294.0,26
M4,617.0,1
M5,16614.0,78


* Size is **very skewed**, so it's shown (and should be compared) on a **log scale**: a 2,000 vs 4,000 sq ft
  difference should count the same as 20,000 vs 40,000. On that scale the portfolio has one broad peak around
  2,000-8,000 sq ft (median 4,753) and a long tail of big sheds up to 1.2m sq ft.
* The listings fall into two groups:
  * **small: M4 (617 sq ft), M2 (906) and M3 (2,294)**. M4 and M2 are smaller than 99% and 95% of portfolio
    units; M3 is in the lower-middle;
  * **mid-sized: M1 (16,860) and M5 (16,614)**, both larger than about 80% of portfolio units.
* None sits in the peak of the distribution, so the size score will separate them. Section 3.4 builds it.

### 3.3 Pairwise similarity (prompt 1)

**`size_similarity(listing, portfolio)`** compares sizes as a ratio, halving each time one size is
`SIZE_HALF_LIFE_RATIO` (2) times the other. For M3 (2,294 sq ft):


In [29]:
from indurent_matching.similarity import size_similarity

m3 = marketed.loc["M3"]
m3_size = size_similarity(m3, portfolio)
print(f"{m3_size.notna().sum():,} units scored; {m3_size.isna().sum()} NaN (doubtful sizes)")

fig = px.scatter(x=np.log10(portfolio["size_sqft"]), y=m3_size, opacity=0.4, height=380,
                 title=f"Size similarity of every portfolio unit to M3 ({m3['size_sqft']:,.0f} sq ft)",
                 labels={"x": "Portfolio unit size (sq ft, log scale)", "y": "Size similarity to M3"})
fig.add_vline(x=np.log10(m3["size_sqft"]), line_dash="dash", line_color="firebrick")
fig.update_xaxes(tickvals=np.log10(ticks), ticktext=[f"{t:,}" for t in ticks])
fig.show()

1,240 units scored; 10 NaN (doubtful sizes)


In [30]:
sample = portfolio.loc[[*m3_size.dropna().sort_values().index[[0, 400, 800, 1100, -1]]],
                       ["industrial_estate_name", "unit_name", "size_sqft"]].round(0)
sample["times M3's size"] = (sample["size_sqft"] / m3["size_sqft"]).round(2)
sample["size_similarity to M3"] = m3_size[sample.index].round(2)
sample

,industrial_estate_name,unit_name,size_sqft,times M3's size,size_similarity to M3
property_id,,,,,
1179,Granite Oaks 47,Unit 1,1195451.0,521.01,0.00
1219,Topaz Square 51,Unit 1,9253.0,4.03,0.25
755,Teal Court 89,Unit 14,1260.0,0.55,0.55
13,Ivory Wharf 66,Unit 13,2751.0,1.20,0.83
986,Teal Grove 11,Unit 6,2295.0,1.00,1.00


* Every unit sits on one symmetric curve on the log scale: 1 at M3's size (2,294 sq ft), 0.5 at half (about
  1,150) and at double (about 4,600), falling towards 0 for the big sheds.
* Because sizes are compared as a ratio, a unit at half M3's size scores the same as one at twice it.
* The 10 doubtful sizes are NaN here, and count as 0 when the dimensions are combined.

### 3.4 Homogeneity (prompt 2): a size score

The same idea as the age score: count portfolio units in **size bands**, and divide by the count in the most frequent
band, so the most common size scores 1. Because size is compared on a log scale, the bands **double** in width
(500-1k, 1k-2k, 2k-4k sq ft, ...), so each band is the same width on the log scale.

In [31]:
edges = 500 * 2.0 ** np.arange(0, 13)  # 500, 1,000, 2,000, ... 2,048,000 sq ft


def short(sqft):
    """Readable band edge: 500, 1k, 2k, ... 512k, 1m, 2m."""
    if sqft < 1_000:
        return f"{sqft:.0f}"
    return f"{sqft / 1e6:.0f}m" if sqft >= 1e6 else f"{sqft / 1e3:.0f}k"


band_labels = [f"{short(lo)}-{short(hi)}" for lo, hi in zip(edges[:-1], edges[1:])]
size_band = pd.cut(size, edges, right=False, labels=band_labels)
per_band = size_band.value_counts(sort=False)
band_score = per_band / per_band.max()

fig = px.bar(x=band_labels, y=band_score.values, height=380,
             title="Score per size band: units in the band / units in the most frequent band",
             labels={"x": "Size band (sq ft)", "y": "Score"})
fig.update_xaxes(type="category")
fig.show()
pd.DataFrame({"units": per_band, "score": band_score.round(2)}).T

size_sqft,500-1k,1k-2k,2k-4k,4k-8k,8k-16k,16k-32k,32k-64k,64k-128k,128k-256k,256k-512k,512k-1m,1m-2m
units,75.00,205.00,265.0,252.00,166.00,105.0,81.00,39.00,27.0,17.00,7.00,1.0
score,0.28,0.77,1.0,0.95,0.63,0.4,0.31,0.15,0.1,0.06,0.03,0.0


In [32]:
listing_size["band"] = pd.cut(listing_size["size_sqft"], edges, right=False, labels=band_labels)
listing_size["size score"] = listing_size["band"].map(band_score).astype(float).round(2)
listing_size

,size_sqft,% of portfolio units smaller,band,size score
listing_id,,,,
M1,16860.0,79,16k-32k,0.40
M2,906.0,5,500-1k,0.28
M3,2294.0,26,2k-4k,1.00
M4,617.0,1,500-1k,0.28
M5,16614.0,78,16k-32k,0.40


* The most frequent band is **2k-4k sq ft** (265 units), so it scores 1; 1k-8k sq ft all score 0.77 or more, and
  the score falls away for big sheds (0.10 or less above 128k).
* The listings: **M3 1.00** (in the most frequent band); **M1 and M5 0.40**; **M2 and M4 0.28**.
* **A boundary problem, unlike age.** M1 (16,860) and M5 (16,614) sit just above the 16k edge: at 15,900 sq ft
  they'd score 0.63, not 0.40. For age the jump at 1970 was in the data; here the distribution slopes smoothly,
  so the jump is made by the bands. Bands this wide (each a doubling) also lump M2 (906) and M4 (617) together,
  though M4 is far rarer: only 1% of portfolio units are smaller than it, against 5% for M2.

**Alternative: a moving band centred on the listing.** Count portfolio units within a doubling-wide window
*centred on the listing's own size* (from size / 1.41 to size x 1.41), and divide by the largest such count
anywhere in the portfolio. Same idea and same width, but no fixed edges to fall either side of.

In [33]:
sorted_size = np.sort(size.to_numpy())


def units_in_window(sqft):
    """Portfolio units of a similar size to sqft: from sqft / 1.41 to sqft x 1.41 (the largest twice the smallest)."""
    return np.searchsorted(sorted_size, sqft * np.sqrt(2)) - np.searchsorted(sorted_size, sqft / np.sqrt(2))


# The curve is drawn along a fine grid of sizes; the denominator is the fullest window centred on a portfolio unit's
# own size (as in similarity.size_homogeneity), so it doesn't depend on the grid.
grid = np.geomspace(size.min(), size.max(), 2000)
window_counts = np.array([units_in_window(g) for g in grid])
unit_counts = np.array([units_in_window(s) for s in sorted_size])
most_frequent = unit_counts.max()

fig = px.line(x=np.log10(grid), y=window_counts / most_frequent, height=380,
              title="Moving-window size score (dashed lines: listings)",
              labels={"x": "Size (sq ft, log scale)", "y": "Score"})
fig.update_xaxes(tickvals=np.log10(ticks), ticktext=[f"{t:,}" for t in ticks])
for sqft in listing_size["size_sqft"]:
    fig.add_vline(x=np.log10(sqft), line_dash="dash", line_color="firebrick")
fig.show()

print(f"Fullest window: {most_frequent} units, centred on a unit of {sorted_size[unit_counts.argmax()]:,.0f} sq ft")
# Score the unrounded sizes (listing_size shows them rounded to whole sq ft).
listing_size["moving-window score"] = [
    round(units_in_window(s) / most_frequent, 2) for s in listing_physical["size_sqft"]
]
listing_size[["size_sqft", "band", "size score", "moving-window score"]]

Fullest window: 298 units, centred on a unit of 4,207 sq ft


,size_sqft,band,size score,moving-window score
listing_id,,,,
M1,16860.0,16k-32k,0.40,0.41
M2,906.0,500-1k,0.28,0.38
M3,2294.0,2k-4k,1.00,0.87
M4,617.0,500-1k,0.28,0.17
M5,16614.0,16k-32k,0.40,0.41


* The moving-window curve is **smooth**: no jumps, so a listing's score doesn't depend on which side of an edge
  it falls. It peaks around 4,200 sq ft, the middle of the portfolio's broad peak.
* **M1 and M5 (0.41)** score about the same as with bands, but now for the right reason: their score wouldn't
  change at 15,900 sq ft.
* **M2 (0.38) and M4 (0.17) are now told apart**: M4, smaller than 99% of the portfolio, scores half of M2.
* **M3 (0.87)** is just off the peak, rather than sitting in the most frequent band at exactly 1.
* For age, fixed bands were fine because the distribution was flat and its one jump (1970) was real. Size
  slopes steadily, so fixed bands distort it more; the moving window is the better fit here.

### 3.5 Size and eaves height together

With doors and volume left out, the physical dimension comes down to **two measures: size and eaves height**. Eaves height
is the average of min and max where both are recorded, or whichever one is. The scatter shows the 418 portfolio
units with both, and the listings: **M4 and M5** have both (stars); **M1, M2 and M3** have size only, so they're
shown as dashed lines at their size.

In [34]:
import plotly.graph_objects as go


def eaves_height(df):
    """Average of min and max eaves where both are recorded, otherwise whichever one is."""
    return df[["min_eaves_m", "max_eaves_m"]].mean(axis=1)


both = physical.assign(eaves_m=eaves_height(physical)).dropna(subset=["size_sqft", "eaves_m"])
both = both.join(portfolio[["industrial_estate_name", "unit_name"]])
listings_2d = listing_physical.assign(eaves_m=eaves_height(listing_physical))

fig = px.scatter(both, x=np.log10(both["size_sqft"]), y="eaves_m", hover_name="industrial_estate_name",
                 hover_data={"unit_name": True, "size_sqft": ":,.0f"}, opacity=0.5, height=480,
                 title="Portfolio units with size and eaves height, and the listings",
                 labels={"x": "Size (sq ft, log scale)", "eaves_m": "Eaves height (m)"})
fig.update_traces(marker={"size": 6}, name="portfolio unit", showlegend=True)
with_eaves = listings_2d.dropna(subset=["eaves_m"])
fig.add_trace(go.Scatter(x=np.log10(with_eaves["size_sqft"]), y=with_eaves["eaves_m"], mode="markers+text",
                         text=with_eaves.index, textposition="top center", name="listing (size and eaves)",
                         marker={"symbol": "star", "size": 16, "color": "firebrick"}))
for sqft in listings_2d.loc[listings_2d["eaves_m"].isna(), "size_sqft"]:
    fig.add_vline(x=np.log10(sqft), line_dash="dash", line_color="firebrick", opacity=0.5)
fig.update_xaxes(tickvals=np.log10(ticks), ticktext=[f"{t:,}" for t in ticks])
fig.show()

print(f"{len(both)} portfolio units with size and eaves; "
      f"rank correlation of eaves with size: {both['eaves_m'].corr(both['size_sqft'], method='spearman'):.2f}")
for listing_id, row in with_eaves.iterrows():
    peers = both[(both["size_sqft"] > row["size_sqft"] / 1.41) & (both["size_sqft"] < row["size_sqft"] * 1.41)]
    print(f"{listing_id}: {row['size_sqft']:,.0f} sq ft, {row['eaves_m']:.1f} m eaves; {len(peers)} portfolio units "
          f"of similar size have eaves, median {peers['eaves_m'].median():.1f} m "
          f"(10th-90th percentile {peers['eaves_m'].quantile(0.1):.1f}-{peers['eaves_m'].quantile(0.9):.1f} m)")

418 portfolio units with size and eaves; rank correlation of eaves with size: 0.57
M4: 617 sq ft, 7.0 m eaves; 6 portfolio units of similar size have eaves, median 4.5 m (10th-90th percentile 4.3-6.5 m)
M5: 16,614 sq ft, 5.3 m eaves; 45 portfolio units of similar size have eaves, median 6.3 m (10th-90th percentile 5.1-8.6 m)


* **Eaves rise with size** (rank correlation 0.57), but with a wide spread: units of the same size can differ
  by several metres. So eaves add real information beyond size, which is why it's worth keeping as the second
  dimension.
* **M4 is tall for its size**: 7 m eaves on 617 sq ft, above all but a few of the 6 portfolio units of similar
  size with eaves recorded (median 4.5 m, 90th percentile 6.5 m). With only 6 comparable units, though, that
  judgement rests on very little data. (M4's 16 doors were already flagged as implausible for its size.)
* **M5 is a little low for its size**: 5.3 m against a median of 6.3 m among 45 similar-sized units, but well
  within their range (5.1-8.6 m).
* **M1, M2 and M3** (dashed lines) can only be placed on size.
* **Decision: score on size only, and report eaves as context.** Only M4 and M5 have eaves, so a size-and-eaves
  score would put them on a different basis from M1, M2 and M3. Scoring size alone keeps all five comparable;
  eaves still appear in answers, e.g. *"M4's eaves (7 m) are tall for a unit of its size."*

### 3.6 Decisions for `similarity.py`

| Topic | Decision | Why |
|---|---|---|
| Physical score | **Size only**, scored with a moving window: portfolio units of a similar size to the listing (size / 1.41 to size x 1.41, so the largest is twice the smallest), divided by the most units any portfolio unit has of a similar size (0-1) | Size is the only physical field every listing has, so all five are scored on the same basis. The moving window has no band edges: fixed bands made a jump at 16k sq ft that isn't in the data, and lumped M2 with the much rarer M4 |
| Scale | Log (each window spans a doubling) | Size runs from 505 to 1.2m sq ft; a 2k vs 4k difference should count the same as 20k vs 40k |
| Doubtful sizes | Left out of the counts | The 10 units under 500 sq ft (EDA 2.1) |
| Eaves height | Context only: average of min and max, or whichever is recorded, compared with similar-sized portfolio units | Only M4 and M5 have it; scoring it would put them on a different basis. Still useful in answers ("tall for its size") |
| Doors | Context only | 39 usable portfolio counts, one usable listing count; they rise with size anyway |
| Volume | Context only | 99% correlated with size, needs eaves, and treats a big low shed and a small tall one as the same |
| Not used | Fixed size bands | Edge jumps made by the bands, and too coarse at the small end |